<a href="https://colab.research.google.com/github/edwintorrecilla-create/control-cedulas/blob/main/almacen_insumos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import shutil
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import PieChart, Reference

def procesar_movimientos_sap():
    folder_entrada = 'entrada'
    folder_salida = 'salida'
    os.makedirs(folder_entrada, exist_ok=True)
    os.makedirs(folder_salida, exist_ok=True)

    input_file = os.path.join(folder_entrada, 'Movimientos Ingresos Almacen SAP.xlsx')
    output_file = os.path.join(folder_salida, 'Análisis_Movimientos_Ingresos_SAP_Con_HT.xlsx')

    if not os.path.exists(input_file):
        if os.path.exists('Movimientos Ingresos Almacen SAP.xlsx'):
            shutil.copy('Movimientos Ingresos Almacen SAP.xlsx', input_file)
        else:
            print(f"Coloca el archivo inicial en: {input_file}")
            return

    # 1. Cargar archivo
    df = pd.read_excel(input_file)
    if 'Código del Articulo' not in df.columns:
        df = pd.read_excel(input_file, header=1)

    df.columns = df.columns.astype(str).str.strip()

    code_col = [c for c in df.columns if 'Articulo' in c or 'Artículo' in c][0]
    total_col = [c for c in df.columns if 'Total' in c][0]

    # Extraer Prefijo de 2 caracteres
    df['Código Prefijo'] = df[code_col].astype(str).str.strip().str[:2].str.upper()

    # Omitir categorías MP (Materias Primas) y AP (Accesorios de Ensamble)
    prefijos_omitidos = ['MP', 'AP']
    df_filtered = df[~df['Código Prefijo'].isin(prefijos_omitidos)].copy()

    # =======================================================
    # MAPEO ACTUALIZADO: INCLUYE HT = HERRAMIENTAS
    # =======================================================
    cat_map = {
        'RT': 'Repuestos de Maquina',
        'DT': 'Dotación',
        'CN': 'Consumibles',
        'SG': 'EPPS',
        'HT': 'Herramientas'
    }

    df_filtered['Categoría'] = df_filtered['Código Prefijo'].map(cat_map).fillna('Otros')
    df_filtered[total_col] = pd.to_numeric(df_filtered[total_col], errors='coerce').fillna(0)

    # Agrupar datos por categoría
    summary_df = df_filtered.groupby('Categoría', as_index=False).agg(
        Num_Items=(total_col, 'count'),
        Total_Suma=(total_col, 'sum')
    )

    total_general = summary_df['Total_Suma'].sum()
    summary_df['Porcentaje'] = summary_df['Total_Suma'] / total_general if total_general > 0 else 0
    summary_df = summary_df.sort_values(by='Total_Suma', ascending=False).reset_index(drop=True)

    # 2. Generar Excel
    wb = openpyxl.Workbook()

    ws_resumen = wb.active
    ws_resumen.title = "Resumen Categorías"
    ws_resumen.views.sheetView[0].showGridLines = True

    ws_detalle = wb.create_sheet(title="Detalle Categorizado")
    ws_detalle.views.sheetView[0].showGridLines = True

    # Estilos visuales
    font_title = Font(name='Segoe UI', size=15, bold=True, color='1B365D')
    font_subtitle = Font(name='Segoe UI', size=10, italic=True, color='555555')
    font_header = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
    font_data = Font(name='Segoe UI', size=10, color='333333')
    font_total = Font(name='Segoe UI', size=11, bold=True, color='1B365D')

    fill_header = PatternFill(start_color='1B365D', fill_type='solid')
    fill_zebra = PatternFill(start_color='F4F7FA', fill_type='solid')
    fill_total = PatternFill(start_color='E8EEF5', fill_type='solid')

    border_thin = Border(
        left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
    )
    border_total = Border(
        top=Side(style='thin', color='1B365D'), bottom=Side(style='double', color='1B365D')
    )

    # Encabezados en Resumen
    ws_resumen['B2'] = "RESUMEN DE INGRESOS A ALMACÉN (INCLUYE HERRAMIENTAS - HT)"
    ws_resumen['B2'].font = font_title
    ws_resumen['B3'] = "Categorización automática incluyendo Herramientas (HT) y excluyendo MP y AP"
    ws_resumen['B3'].font = font_subtitle

    headers = ["Categoría", "N° Movimientos", "Total ($)", "% Participación"]
    for col_idx, h in enumerate(headers, start=2):
        cell = ws_resumen.cell(row=5, column=col_idx, value=h)
        cell.font = font_header
        cell.fill = fill_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    start_row = 6
    for idx, row in summary_df.iterrows():
        curr_row = start_row + idx
        ws_resumen.cell(row=curr_row, column=2, value=row['Categoría']).font = font_data
        ws_resumen.cell(row=curr_row, column=3, value=row['Num_Items']).font = font_data
        ws_resumen.cell(row=curr_row, column=4, value=row['Total_Suma']).font = font_data
        ws_resumen.cell(row=curr_row, column=5, value=row['Porcentaje']).font = font_data

        ws_resumen.cell(row=curr_row, column=2).alignment = Alignment(horizontal='left')
        ws_resumen.cell(row=curr_row, column=3).alignment = Alignment(horizontal='right')
        ws_resumen.cell(row=curr_row, column=3).number_format = '#,##0'
        ws_resumen.cell(row=curr_row, column=4).alignment = Alignment(horizontal='right')
        ws_resumen.cell(row=curr_row, column=4).number_format = '"$"#,##0.00'
        ws_resumen.cell(row=curr_row, column=5).alignment = Alignment(horizontal='right')
        ws_resumen.cell(row=curr_row, column=5).number_format = '0.0%'

        if idx % 2 == 1:
            for c in range(2, 6):
                ws_resumen.cell(row=curr_row, column=c).fill = fill_zebra

        for c in range(2, 6):
            ws_resumen.cell(row=curr_row, column=c).border = border_thin

    end_row = start_row + len(summary_df) - 1
    total_row = end_row + 1

    # Fila de Totales
    ws_resumen.cell(row=total_row, column=2, value="TOTAL GENERAL").font = font_total
    ws_resumen.cell(row=total_row, column=3, value=f"=SUM(C{start_row}:C{end_row})").font = font_total
    ws_resumen.cell(row=total_row, column=4, value=f"=SUM(D{start_row}:D{end_row})").font = font_total
    ws_resumen.cell(row=total_row, column=5, value=f"=SUM(E{start_row}:E{end_row})").font = font_total

    ws_resumen.cell(row=total_row, column=2).alignment = Alignment(horizontal='left')
    ws_resumen.cell(row=total_row, column=3).number_format = '#,##0'
    ws_resumen.cell(row=total_row, column=4).number_format = '"$"#,##0.00'
    ws_resumen.cell(row=total_row, column=5).number_format = '0.0%'

    for c in range(2, 6):
        ws_resumen.cell(row=total_row, column=c).fill = fill_total
        ws_resumen.cell(row=total_row, column=c).border = border_total

    # Crear Gráfico de Torta
    pie = PieChart()
    pie.title = "Distribución de Ingresos por Categoría"
    labels = Reference(ws_resumen, min_col=2, min_row=start_row, max_row=end_row)
    data = Reference(ws_resumen, min_col=4, min_row=5, max_row=end_row)

    pie.add_data(data, titles_from_data=True)
    pie.set_categories(labels)
    pie.width = 16
    pie.height = 11

    ws_resumen.add_chart(pie, "G5")

    ws_resumen.column_dimensions['A'].width = 3
    ws_resumen.column_dimensions['B'].width = 28
    ws_resumen.column_dimensions['C'].width = 16
    ws_resumen.column_dimensions['D'].width = 22
    ws_resumen.column_dimensions['E'].width = 16

    # Llenar Pestaña Detalle
    cols = list(df_filtered.columns)
    if 'Categoría' in cols:
        cols.remove('Categoría')
    art_idx = cols.index(code_col)
    cols.insert(art_idx + 1, 'Categoría')
    df_detail = df_filtered[cols]

    ws_detalle['A1'] = "DETALLE DE MOVIMIENTOS INGRESOS ALMACÉN SAP"
    ws_detalle['A1'].font = font_title

    for col_idx, col_name in enumerate(df_detail.columns, start=1):
        cell = ws_detalle.cell(row=3, column=col_idx, value=str(col_name))
        cell.font = font_header
        cell.fill = fill_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    for row_idx, row in df_detail.iterrows():
        r = row_idx + 4
        for col_idx, col_name in enumerate(df_detail.columns, start=1):
            val = row[col_name]
            cell = ws_detalle.cell(row=r, column=col_idx, value=val)
            cell.font = font_data
            cell.border = border_thin

            if 'Fecha' in col_name and pd.notna(val):
                cell.number_format = 'YYYY-MM-DD'
                cell.alignment = Alignment(horizontal='center')
            elif col_name in ['Vr Unitario', 'Total']:
                cell.number_format = '"$"#,##0.00'
                cell.alignment = Alignment(horizontal='right')
            elif col_name == 'Categoría':
                cell.alignment = Alignment(horizontal='center')
                cell.font = Font(name='Segoe UI', size=10, bold=True, color='1B365D')

    for col in ws_detalle.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws_detalle.column_dimensions[col_letter].width = max(max_len + 3, 12)

    wb.save(output_file)
    print(f"Proceso finalizado. Archivo generado en: {output_file}")

if __name__ == '__main__':
    procesar_movimientos_sap()

Proceso finalizado. Archivo generado en: salida/Análisis_Movimientos_Ingresos_SAP_Con_HT.xlsx
